# Run `amir/score.py` in Colab

Open this notebook in Google Colab, then choose `Runtime -> Change runtime type` and select a GPU runtime.

This workflow is wired for the `amir/transcripts` corpus. GPU is supported through PyTorch + Transformers. TPU runtimes can still open the notebook, but `score.py` currently falls back to CPU unless you add `torch-xla` support.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ngafney/garda-spring26.git"
REPO_DIR = Path("/content/garda-spring26") if IN_COLAB else Path.cwd()
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_REPO_DIR = Path("/content/drive/MyDrive/garda-spring26")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = GOOGLE_DRIVE_REPO_DIR

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")

Working directory: /content/garda-spring26


In [2]:
%pip install -q pandas tqdm torch transformers tomli

In [3]:
import os
import sys
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if os.environ.get("COLAB_TPU_ADDR"):
    print("TPU runtime detected. score.py will still use CPU unless torch-xla support is added.")

Python: 3.12.13
CUDA available: True
CUDA device count: 1
GPU: Tesla T4


## Parameters

Set `LIMIT = None` for the full corpus. There are about 9,951 transcript files in `amir/transcripts`, so a full run can take a long time even on Colab GPU.

Use `PATTERN` to score a subset first, for example `"amazon"` or `"nvidia"`.

If you already have `amir/transcripts_metadata.csv`, leave `REBUILD_METADATA = False` and the notebook will use that file.

In [4]:
TRANSCRIPTS_DIR = REPO_DIR / "amir" / "transcripts"
OUTPUT_DIR = REPO_DIR / "amir" / "outputs"
METADATA_CSV = REPO_DIR / "amir" / "transcripts_metadata.csv"
OUTPUT_CSV = OUTPUT_DIR / "transcript_scores.csv"
SCORE_DB = OUTPUT_DIR / "score_cache.sqlite"

LIMIT = 5
PATTERN = None
FORCE = True
REBUILD_METADATA = False
VERBOSE = 1

print(TRANSCRIPTS_DIR)
print(METADATA_CSV)
print(OUTPUT_CSV)


/content/garda-spring26/amir/transcripts
/content/garda-spring26/amir/transcripts_metadata.csv
/content/garda-spring26/amir/outputs/transcript_scores.csv


In [5]:
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "amir.score",
    "--transcripts-dir",
    str(TRANSCRIPTS_DIR),
    "--metadata-csv",
    str(METADATA_CSV),
    "--output-csv",
    str(OUTPUT_CSV),
    "--score-db",
    str(SCORE_DB),
]

if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
if PATTERN:
    cmd += ["--pattern", PATTERN]
if REBUILD_METADATA:
    cmd.append("--rebuild-metadata")
if FORCE:
    cmd.append("--force")
cmd += ["-v"] * VERBOSE

print("Running:")
print(" ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"score.py failed with exit code {result.returncode}")


Running:
/usr/bin/python3 amir/score.py --transcripts-dir /content/garda-spring26/amir/transcripts --metadata-csv /content/garda-spring26/amir/transcripts_metadata.csv --output-csv /content/garda-spring26/amir/outputs/transcript_scores.csv --score-db /content/garda-spring26/amir/outputs/score_cache.sqlite --limit 5 --force -v


CalledProcessError: Command '['/usr/bin/python3', 'amir/score.py', '--transcripts-dir', '/content/garda-spring26/amir/transcripts', '--metadata-csv', '/content/garda-spring26/amir/transcripts_metadata.csv', '--output-csv', '/content/garda-spring26/amir/outputs/transcript_scores.csv', '--score-db', '/content/garda-spring26/amir/outputs/score_cache.sqlite', '--limit', '5', '--force', '-v']' returned non-zero exit status 1.

In [ ]:
import pandas as pd

scores = pd.read_csv(OUTPUT_CSV)
print(f"Rows written: {len(scores):,}")
display(scores.head())